# Experiment 4.4.2 — Stage-2 recurrence topology

Analysis-only notebook. It compares the reused Exp4.4.1 FF/Dense controls with the newly trained Diagonal recurrence at 250 and 500 ms. Primary architecture metric: **Uend + Linear BA**.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find writingRing repository root')

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_4_4_2_recurrence_topology' / 'stage2_recurrence_topology_v1'
EXPECTED_SEEDS = (11, 23, 37, 53, 71)
EXPECTED_TAUS = (250, 500)
EXPECTED_TOPOLOGIES = ('ff', 'diagonal', 'dense')
required = ('runs.csv', 'summary.csv', 'paired_effects.csv', 'paired_effects_summary.csv', 'manifest.json')
missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing finalized Exp4.4.2 artifacts: {missing}')

runs = pd.read_csv(ARTIFACT_DIR / 'runs.csv')
summary = pd.read_csv(ARTIFACT_DIR / 'summary.csv')
effects = pd.read_csv(ARTIFACT_DIR / 'paired_effects.csv')
effects_summary = pd.read_csv(ARTIFACT_DIR / 'paired_effects_summary.csv')
manifest = json.loads((ARTIFACT_DIR / 'manifest.json').read_text())

test = runs[runs['split'] == 'test']
assert set(test['seed']) == set(EXPECTED_SEEDS)
assert set(test['tau_mem_ms']) == set(EXPECTED_TAUS)
assert set(test['topology']) == set(EXPECTED_TOPOLOGIES)
assert len(test) == len(EXPECTED_SEEDS) * len(EXPECTED_TAUS) * len(EXPECTED_TOPOLOGIES)
manifest


## Aggregate table
The table keeps all three readouts visible so topology effects on endpoint memory are not conflated with spike-count accessibility.


In [ ]:
cols = [
    'topology', 'tau_mem_ms',
    'uend_linear_ba_mean', 'uend_linear_ba_std',
    'hidden_whole_count_linear_ba_mean', 'hidden_whole_count_linear_ba_std',
    'output_whole_count_ba_mean', 'output_whole_count_ba_std',
    'uend_minus_hidden_count_ba_mean', 'uend_minus_output_count_ba_mean',
]
display(summary[cols].sort_values(['tau_mem_ms', 'topology']).reset_index(drop=True))


## Uend memory quality by recurrence topology
A Diagonal result near Dense would suggest neuron-wise recurrent persistence is sufficient. A clear Dense advantage would support population-level recurrent mixing, subject to the parameter-count caveat.


In [ ]:
order = list(EXPECTED_TOPOLOGIES)
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for tau in EXPECTED_TAUS:
    frame = summary[summary['tau_mem_ms'] == tau].set_index('topology').loc[order]
    ax.errorbar(x, frame['uend_linear_ba_mean'], yerr=frame['uend_linear_ba_std'], marker='o', capsize=4, label=f'{tau} ms')
ax.set_xticks(x, order)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Stage-2 endpoint memory: Uend + Linear')
ax.set_ylim(0, 0.7)
ax.legend(title='Stage-2 tau')
ax.grid(axis='y', alpha=0.25)
plt.show()


## Readout matrix
For each tau, compare endpoint state, hidden spike count, and final Output WholeCount.


In [ ]:
readouts = {
    'Uend + Linear': ('uend_linear_ba_mean', 'uend_linear_ba_std'),
    'HiddenCount + Linear': ('hidden_whole_count_linear_ba_mean', 'hidden_whole_count_linear_ba_std'),
    'Output WholeCount': ('output_whole_count_ba_mean', 'output_whole_count_ba_std'),
}
for tau in EXPECTED_TAUS:
    frame = summary[summary['tau_mem_ms'] == tau].set_index('topology').loc[order]
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for label, (mean_col, std_col) in readouts.items():
        ax.errorbar(x, frame[mean_col], yerr=frame[std_col], marker='o', capsize=3, label=label)
    ax.set_xticks(x, order)
    ax.set_ylabel('Test balanced accuracy')
    ax.set_ylim(0, 0.7)
    ax.set_title(f'Readout matrix — Stage-2 tau = {tau} ms')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    plt.show()


## Paired topology effects
Paired effects use the same seed/split/scaling protocol. Focus on `uend_linear` first, then inspect whether the same gain reaches spike-count readouts.


In [ ]:
uend_effects = effects_summary[effects_summary['readout'] == 'uend_linear'].copy()
display(uend_effects.sort_values(['tau_mem_ms', 'effect']).reset_index(drop=True))
fig, ax = plt.subplots(figsize=(9, 5.2))
for effect, frame in uend_effects.groupby('effect'):
    frame = frame.sort_values('tau_mem_ms')
    ax.errorbar(frame['tau_mem_ms'], frame['mean'], yerr=frame['std'], marker='o', capsize=4, label=effect)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Stage-2 tau_mem (ms)')
ax.set_ylabel('Paired delta BA')
ax.set_title('Paired recurrence-topology effects on Uend memory')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## State-to-spike gap and dynamics
Smaller positive `Uend - OutputCount` means more of the endpoint-state information is already accessible through the deployment-facing count readout. Tail activity is shown as a stability diagnostic, not as a memory score.


In [ ]:
diagnostic_cols = [
    'topology', 'tau_mem_ms',
    'uend_minus_hidden_count_ba_mean', 'uend_minus_output_count_ba_mean',
    'state_events_per_neuron_second_mean', 'output_events_per_neuron_second_mean',
    'state_tail_event_fraction_mean', 'output_tail_event_fraction_mean',
]
display(summary[diagnostic_cols].sort_values(['tau_mem_ms', 'topology']).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8.5, 5.2))
for topology in order:
    frame = summary[summary['topology'] == topology].sort_values('tau_mem_ms')
    ax.plot(frame['tau_mem_ms'], frame['uend_minus_output_count_ba_mean'], marker='o', label=topology)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Stage-2 tau_mem (ms)')
ax.set_ylabel('Uend BA - Output WholeCount BA')
ax.set_title('State-to-output information gap')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Interpretation boundary
Diagonal recurrence has 128 recurrent parameters while Dense has 16,384. Therefore a Dense advantage is evidence that full population recurrence is effective in this architecture, but it is not yet a parameter-matched proof that cross-neuron mixing itself is the sole cause.
